In [2]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()


In [2]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [3]:
from rag_helper import RAGBase

instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=instructions,
)

In [4]:
answer = assistant.rag("How do I run Ollama locally?")
print(answer)

To run Ollama locally:

1. Install Ollama from [https://ollama.com/download](https://ollama.com/download) for your operating system.
2. Open a terminal and run:
   ```bash
   ollama run llama3
   ```
   This downloads the LLaMA 3 model, starts it locally, and opens a chat-like interface.
3. To test the local server, run:
   ```bash
   curl http://localhost:11434
   ```
   You should see a response like:
   ```json
   {"models": [...]}  
   ```

If the server needs restarting, you can run:
```bash
!nohup ollama serve > nohup.out 2>&1 &
```


In [5]:
messages = [
    {'role': 'user', 'content': 'I just discovered the course. Can I join it'}
]

response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=messages
)

In [6]:
print(response.output_text)

Absolutely — if the course is still open, you can usually join it.

To confirm, I’d recommend checking:
- whether enrollment is still available,
- if there’s a deadline,
- and whether you need an invitation or approval.

If you want, I can also help you write a short message asking to join the course.


In [7]:
def search(query):
    boost_dict = {'question':3.0, 'section':0.5}
    filter_dict = {'course': 'llm-zoomcamp'}

    results = index.search(
            query=query,
            num_results=5,
            boost_dict=boost_dict,
            filter_dict=filter_dict
    )

    return results

In [8]:
query = 'how do I run ollama locally?'

search(query)

[{'id': '1d0b969028',
  'course': 'llm-zoomcamp',
  'section': 'Module 1: RAG',
  'question': 'Ollama: How to install Ollama?',
  'answer': 'First, install Ollama by visiting [https://ollama.com/download](https://ollama.com/download) and choosing your operating system:\n\n- **macOS**: Download the `.pkg` and install it.\n- **Windows**: Download the `.msi` and install it.\n- **Linux**: Run the following command in the terminal:\n\n  ```bash\n  curl -fsSL https://ollama.com/install.sh | sh\n  ```\n\nOnce installed, open a terminal and type:\n\n```bash\nollama run llama3\n```\n\nThis command will:\n\n- Download the LLaMA 3 model (~4GB).\n- Start the model locally.\n- Open a chat-like interface where you can type questions.\n\nTo test the Ollama local server, run the following command:\n\n```bash\ncurl http://localhost:11434\n```\n\nYou should receive a response similar to:\n\n```json\n{"models": [...]}  \n```\n\nThen, install the Python client with:\n\n```bash\npip install ollama\n```\n\n

In [9]:
## We define a search 'tool' for the llm to use. 

## Here we outline what the tool is, give it a name, describe what it does, define it's parameters

search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
} 

In [10]:
messages = [
    {'role': 'user', 'content': 'I just discovered the course. Can I join it'}
]

In [11]:
response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=messages,
    tools=[search_tool] ## we pass the tools here as a list
)

In [12]:
call = response.output[0]

In [13]:
call

ResponseFunctionToolCall(arguments='{"query":"Can I join the course late discovered just found out about course enrollment join late"}', call_id='call_cCuIGpowWwA9fgrLKLdD5Amf', name='search', type='function_call', id='fc_063c7ae33a12ac7b006a314d39847c81a1bb76e11bad6bdfae', namespace=None, status='completed')

In [14]:
import json
args = json.loads(call.arguments)

In [15]:
results = search(**args)

## Sequence of Steps 

1. Make a call to the LLM <-- First call
2. LLM invokes search with the params we provide
3. Store the results of the search
4. Pass the results to the LLM  <-- Second Call
5. LLM uses the results to generate an answer
6. LLM returns the answer


### IMPORTANT
LLM's are stateless so we have to send the entire history in step 4 above or else it won't know what has come before.

In [16]:
result_json = json.dumps(results, indent=2)

In [17]:
function_call_output = {
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": result_json

}

In [18]:
# Append the original message from call
messages.append(call)

In [19]:
messages

[{'role': 'user', 'content': 'I just discovered the course. Can I join it'},
 ResponseFunctionToolCall(arguments='{"query":"Can I join the course late discovered just found out about course enrollment join late"}', call_id='call_cCuIGpowWwA9fgrLKLdD5Amf', name='search', type='function_call', id='fc_063c7ae33a12ac7b006a314d39847c81a1bb76e11bad6bdfae', namespace=None, status='completed')]

In [20]:
## Add the function_call_output so the model knows what we've previously asked
messages.append(function_call_output)

In [21]:
messages

[{'role': 'user', 'content': 'I just discovered the course. Can I join it'},
 ResponseFunctionToolCall(arguments='{"query":"Can I join the course late discovered just found out about course enrollment join late"}', call_id='call_cCuIGpowWwA9fgrLKLdD5Amf', name='search', type='function_call', id='fc_063c7ae33a12ac7b006a314d39847c81a1bb76e11bad6bdfae', namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_cCuIGpowWwA9fgrLKLdD5Amf',
  'output': '[\n  {\n    "id": "74eb249bbf",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "I just discovered the course. Can I still join?",\n    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."\n  },\n  {\n    "id": "977bf7786c",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "Course: I have registered for the LLM Zoomcamp. When c

In [22]:
#Now pass everything to the LLM to generate an informed response
response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=messages,
    tools=[search_tool] ## we pass the tools here as a list
)

In [23]:
print(response.output_text)

Yes — you can still join the course.

If you want a certificate, make sure you submit your project while submissions are still being accepted.


In [24]:
response.usage

ResponseUsage(input_tokens=775, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=32, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=807)

# The Agentic Loop

If what if we want the agent to keep making api calls until it's satisfied? 

To do that, we update the sequence by adding a loop. 

## Sequence of Steps 

1. Make a call to the LLM <-- First call
2. LLM invokes search with the params we provide
3. We invoke the search
4. Send the results back to the LLM (another call)
5. LLM processes teh results 
6. LLM decides to make another tool call
7. Send one more API request
8. Process and return the answer

## Developer Prompt
To make this loop, we use a developer prompt to explicitly tell the the agent to make multiple searches. 

In [25]:
def make_call(call):
    args = json.loads(call.arguments)

    if call.name == 'search':
        result = search(**args)
    
    result_json = json.dumps(result, indent=2)

    json_file = {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json

    }

    return json_file

In [26]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
"""

question = 'I just discovered the course. Can I join it?'

Agents need: 
- a role; we are providing that in the instructions.
- memory; the messages serve that purpose. 
- tools; search-tools are what we're currently using. 

In [27]:
#Now pass everything to the LLM to generate an informed response
response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=messages,
    tools=[search_tool] ## we pass the tools here as a list
)

In [28]:

messages = [
    {'role': 'developer', 'content': instructions},
    {'role': 'user', 'content': question}
]

iterations = 1

while True:
    print(f'iteration #{iterations}')
    has_function_calls = False

    response = openai_client.responses.create(
        model='gpt-5.4-mini',
        input=messages,
        tools=[search_tool] ## we pass the tools here as a list
    )

    messages.extend(response.output)

    for item in response.output:
        if item.type == 'function_call':
            print('function_call:', item.name, item.arguments)
            call_output = make_call(item)
            messages.append(call_output)
            has_function_calls = True

        elif item.type == 'message':
            print('ASSISTANT:')
            last_answer = item.content[0].text
            print(item.content[0].text)
    
    iterations = iterations+1
    if has_function_calls == False:
        break

iteration #1
function_call: search {"query":"join course enrollment discovered the course can I join"}
iteration #2
ASSISTANT:
Yes — you can still join the course. You can start learning right away, and the materials are available.

One important note: if you want a certificate, you need to submit your project while submissions are still being accepted, since certificates are only available for the live cohort.

If you want, I can also help you find the course docs, schedule, or next steps.


In [29]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results, and then perform more searches. 

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
"""

question = 'I just discovered the course. Can I join it?'

In [30]:
def agent_loop(instructions, question, model='gpt-5.4-mini') -> str:

    messages = [
        {'role': 'developer', 'content': instructions},
        {'role': 'user', 'content': question}
    ]

    iterations = 1

    # Agentic Tool Call Loop
    while True:
        print(f'iteration #{iterations}')
        has_function_calls = False

        response = openai_client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool] ## we pass the tools here as a list
        )

        messages.extend(response.output)

        for item in response.output:
            if item.type == 'function_call':
                print('function_call:', item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

            elif item.type == 'message':
                print('ASSISTANT:')
                last_answer = item.content[0].text
                print(item.content[0].text)
        
        iterations = iterations+1
        if has_function_calls == False:
            break

    return last_answer

In [31]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results, and then perform more searches. 

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
"""

question = 'I just discovered the course. Can I join it?'

In [32]:
result = agent_loop(instructions=instructions,model='gpt-5.4-mini', question=question )

iteration #1
function_call: search {"query":"join course enroll registration discovered course can I join"}
function_call: search {"query":"course enrollment how to join discovered course FAQ"}
iteration #2
ASSISTANT:
Yes — you can still join the course.

If you want a certificate, though, you need to submit your project while the course is still accepting submissions. You can also start learning and submitting homework while the form is open, even if you didn’t register earlier.

If you want, I can also help you figure out how to start the course or how the weekly workflow works.


In [33]:
question = "What's Queen's Gambit"
result = agent_loop(instructions=instructions,model='gpt-5.4-mini', question=question )

iteration #1
function_call: search {"query":"Queen's Gambit chess opening definition"}
iteration #2
function_call: search {"query":"Queen's Gambit chess opening move explained"}
iteration #3
function_call: search {"query":"Queen's Gambit chess opening black accepts or declines"}
iteration #4
ASSISTANT:
The Queen’s Gambit is a classic chess opening that starts with:

1. d4 d5  
2. c4

White offers the c-pawn to try to control the center and challenge Black’s pawn on d5.  
If Black accepts the pawn, it’s called the **Queen’s Gambit Accepted**. If Black doesn’t take it, it’s the **Queen’s Gambit Declined**.

If you want, I can also explain:
- the main ideas behind the opening,
- the difference between Accepted and Declined,
- or show a few common move orders.


## Preventing off Topic Responses

Update the instructions to limit the Agent's scope

In [34]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results, and then perform more searches. 

Try to expand your search by using new keywords
based on the results you get from the search.

The question has to be about the course or its logistics, off topic questions shouldn't be answered. If the search returns nothing, it's likely
an off-topic. If you can't answer the question using FAQ, don't do it yourself. Only use the facts from the FAQ database.



At the end, ask if there are other areas that the user wants to explore.
"""

question = "What is the Queen's Gambit?"

In [35]:
result = agent_loop(instructions=instructions,model='gpt-5.4-mini', question=question )

iteration #1
function_call: search {"query":"Queen's Gambit"}
iteration #2
function_call: search {"query":"gambit queen chess opening course FAQ"}
iteration #3
function_call: search {"query":"Queen's Gambit chess opening FAQ course logistic"}
iteration #4
ASSISTANT:
I couldn’t find any course FAQ entry about “the Queen’s Gambit,” and the search results only returned general course logistics for LLM Zoomcamp.

If you meant a course-related term or something specific in the course materials, please share a bit more context and I can try again. Are there other areas you want to explore?


In [37]:
! uv add toyaikit

Resolved 127 packages in 619ms                                       
⠙ Preparing packages... (0/7)                                                   ⠋ Preparing packages... (0/0)                                                   
⠙ Preparing packages... (0/7)-------------------     0 B/21.96 KiB           
⠙ Preparing packages... (0/7)---------- 14.80 KiB/21.96 KiB         
⠙ Preparing packages... (0/7)---------- 14.80 KiB/21.96 KiB         
docstring-parser     ------------------------------ 14.80 KiB/21.96 KiB
⠙ Preparing packages... (0/7)-------------------     0 B/902.15 KiB          
docstring-parser     ------------------------------ 14.80 KiB/21.96 KiB
⠙ Preparing packages... (0/7)-------------------     0 B/902.15 KiB          
docstring-parser     ------------------------------ 14.80 KiB/21.96 KiB
⠙ Preparing packages... (0/7)-------------------     0 B/902.15 KiB          
docstring-parser     ------------------------------ 14.80 KiB/21.96 KiB
toyaikit             ----------

In [46]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [47]:
def search(query:str) -> dict[str,str]:
    """
    Search the FAQ database for the entries matching the given query.

    Args:
        query (str): Query from the user

    Returns:
        dict[str,str]: _description_
    """

    results = index.search(
            query, 
            num_results=5,
            boost_dict={'questions':3.0, 'section':0.5},
            filter_dict={'course': 'llm-zoomcamp'}
        )

    return results
    

In [48]:
agent_tools = Tools()
agent_tools.add_tool(search)

In [49]:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the FAQ database for the entries matching the given query.\n\nArgs:\n    query (str): Query from the user\n\nReturns:\n    dict[str,str]: _description_',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [52]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

In [53]:
runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model='gpt-5.4-mini')
)

In [54]:
result = runner.loop(
    prompt='How do I run Ollama locally?', 
    callback=callback
)

-> Response received


-> Response received


-> Response received


In [55]:
result.cost

CostInfo(input_cost=Decimal('0.00296475'), output_cost=Decimal('0.001449'), total_cost=Decimal('0.00441375'))

In [57]:
#How do we continue the conversation? Run another loop but pass the previous result to 'previous_messages'
result2 = runner.loop(
    prompt='How do I run a different model?', 
    previous_messages=result.all_messages,
    callback=callback
)

-> Response received


-> Response received


In [58]:
runner.run();

-> Response received


-> Response received


-> Response received


-> Response received


-> Response received


-> Response received


Chat ended.
